# makemore part 3: activations, gradients, batchnorm

This is the *Practice* step of `unit_04_makemore_batchnorm.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one markdown cell (what you're aiming at), one cell of stubs,
and one grader cell. The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the makemore repo or the lecture notebook.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *Python / torch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

The grader is handed `globals()` so it can find whatever you have defined so far. Later milestones
add methods to earlier classes with `Class.method = method` where that helps, so you never re-run one giant cell.

In [ ]:
import math
import random

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from test_makemore_batchnorm import grade

%matplotlib inline

## Boilerplate: the dataset

Same as the previous unit: `names.txt`, a context of 3 characters, 80/10/10 split. Given.

In [ ]:
words = open('../data/names.txt').read().splitlines()
chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

BLOCK = 3      # context length
N_EMBD = 10    # embedding size per character
N_HIDDEN = 200 # hidden units in the one-layer MLP of milestones 1-3


def build_dataset(ws):
    X, Y = [], []
    for w in ws:
        ctx = [0] * BLOCK
        for ch in w + '.':
            ix = stoi[ch]
            X.append(ctx)
            Y.append(ix)
            ctx = ctx[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)


random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(vocab_size, Xtr.shape, Ytr.shape)

## Boilerplate: the one-hidden-layer MLP forward pass

The forward pass itself was last unit's lesson. Here it is given, so this unit can be about
what happens *inside* it at step 0. It returns the three tensors you'll be staring at.

In [ ]:
def forward_manual(params, Xb):
    """params = [C, W1, b1, W2, b2]. Xb: (B, BLOCK) int.
    Returns (hpreact, h, logits): the hidden pre-activations, the tanh outputs, the logits."""
    C, W1, b1, W2, b2 = params
    emb = C[Xb]                                   # (B, BLOCK, N_EMBD)
    embcat = emb.view(emb.shape[0], -1)           # (B, BLOCK * N_EMBD)
    hpreact = embcat @ W1 + b1                    # (B, N_HIDDEN)
    h = torch.tanh(hpreact)                       # (B, N_HIDDEN)
    logits = h @ W2 + b2                          # (B, vocab_size)
    return hpreact, h, logits


def batch(X, Y, n=32, g=None):
    ix = torch.randint(0, X.shape[0], (n,), generator=g)
    return X[ix], Y[ix]

## Milestone 1 — the loss at init

Before a network has seen any data, its loss is a number you can compute on a napkin.
Two things to write: that number, and an initialization whose loss actually lands on it.

The shapes are fixed by the grader: `C (27, 10)`, `W1 (30, 200)`, `b1 (200,)`, `W2 (200, 27)`, `b2 (27,)`.

In [ ]:
def uniform_loss(n_classes):
    """The cross-entropy loss of a model that assigns equal probability to every
    one of n_classes classes. Returns a float."""
    raise NotImplementedError


def init_params(g, vocab_size=vocab_size, n_embd=N_EMBD, block_size=BLOCK, n_hidden=N_HIDDEN):
    """Return [C, W1, b1, W2, b2], all with requires_grad=True, drawn with the
    generator g (use torch.randn(..., generator=g) for every random tensor).

    Shapes: C (vocab_size, n_embd), W1 (block_size*n_embd, n_hidden), b1 (n_hidden,),
            W2 (n_hidden, vocab_size), b2 (vocab_size,).

    Contract for this milestone: the loss at step 0 on a random batch is close to
    uniform_loss(vocab_size). Milestones 2 and 3 tighten the contract further; you
    will come back and edit this function."""
    raise NotImplementedError

In [ ]:
g = torch.Generator().manual_seed(2147483647)
params = init_params(g)
Xb, Yb = batch(Xtr, Ytr, 256, g)
hpreact, h, logits = forward_manual(params, Xb)
print('loss at init:', F.cross_entropy(logits, Yb).item(), '  uniform:', uniform_loss(vocab_size))

In [ ]:
grade(globals(), upto=1)

## Milestone 2 — the saturated tanh

Now look one layer earlier. Three small diagnostics to write, then `init_params` has to pass them.

The plot below is the lecture's: the hidden activations as a histogram, and the batch as an image
where white means "this unit, on this example, is flat".

In [ ]:
def saturation_fraction(h, thresh=0.99):
    """h: (B, n_hidden) tanh outputs. Return the fraction (a float in [0, 1]) of
    ALL entries with |h| > thresh."""
    raise NotImplementedError


def tanh_local_grad(h):
    """h: any-shape tensor of tanh OUTPUTS (h = tanh(x)). Return d tanh(x) / dx,
    same shape, expressed using h only."""
    raise NotImplementedError


def dead_units(h, thresh=0.99):
    """h: (B, n_hidden). Return a bool tensor of shape (n_hidden,): True for a unit
    that is saturated (|h| > thresh) on EVERY example in the batch."""
    raise NotImplementedError

In [ ]:
# Plumbing: look at the hidden layer at init.
g = torch.Generator().manual_seed(2147483647)
params = init_params(g)
Xb, Yb = batch(Xtr, Ytr, 32, g)
hpreact, h, logits = forward_manual(params, Xb)

fig, ax = plt.subplots(1, 3, figsize=(16, 3))
ax[0].hist(hpreact.detach().view(-1).tolist(), 50); ax[0].set_title(f'hpreact  std={hpreact.std():.2f}')
ax[1].hist(h.detach().view(-1).tolist(), 50); ax[1].set_title('h = tanh(hpreact)')
ax[2].imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest'); ax[2].set_title('white = saturated')
plt.show()
print('saturated fraction:', saturation_fraction(h), '  dead units:', int(dead_units(h).sum()))

In [ ]:
grade(globals(), upto=2)

## Milestone 3 — the init scale, by principle instead of by hand

You have been picking constants. Derive the one the constants were approximating, then
*measure* it: build a stack of random layers, push a standard-normal input through, and record
the std of the output at every layer. If the scale is right, the numbers don't drift.

Then go back to `init_params` and use `kaiming_std` for `W1`. The tanh gain is `5/3`.

In [ ]:
def kaiming_std(fan_in, gain=1.0):
    """The std to draw a (fan_in, fan_out) weight matrix with so that y = x @ W
    preserves the std of a standard-normal x, times gain. Returns a float."""
    raise NotImplementedError


def activation_std_through_stack(g, depth, width, gain, nonlin=None, n=1000):
    """Start from x ~ N(0, 1) of shape (n, width) drawn with generator g. Apply
    `depth` layers, each y = nonlin(x @ W) (or just x @ W when nonlin is None), with
    W a fresh (width, width) randn(generator=g) scaled by kaiming_std(width, gain).
    Return a list of `depth` floats: the std of each layer's output, in order."""
    raise NotImplementedError

In [ ]:
# Plumbing: the drift plot. Three curves: linear with gain 1, tanh with gain 1, tanh with gain 5/3.
g = torch.Generator().manual_seed(0)
for label, gain, nl in [('linear, gain 1', 1.0, None), ('tanh, gain 1', 1.0, torch.tanh), ('tanh, gain 5/3', 5/3, torch.tanh)]:
    plt.plot(activation_std_through_stack(g, 10, 200, gain, nl), marker='o', label=label)
plt.xlabel('layer'); plt.ylabel('std of output'); plt.legend(); plt.show()

In [ ]:
grade(globals(), upto=3)

## Milestone 4 — BatchNorm1d, from scratch

The wall. You fought for pre-activations that are roughly unit gaussian at init; this layer
*makes* them unit gaussian every step and lets the network learn a scale and shift on top.
The contract is `torch.nn.BatchNorm1d(dim, eps=1e-5, momentum=0.1)`: your layer must agree
with it numerically on the same input, in both modes, forward and backward.

**BatchNorm1d(dim, eps=1e-5, momentum=0.1)**
- Holds: `gamma` `(dim,)` ones and `beta` `(dim,)` zeros, the two learnable tensors;
  `running_mean` `(dim,)` zeros and `running_var` `(dim,)` ones, buffers that never require
  grad; `training`, a bool that is `True` after construction; `out`, the last output.
- `__call__(x)`, `x: (B, dim)`: `out = gamma * (x - mean) / sqrt(var + eps) + beta`. With
  `training=True`, `mean`/`var` are this batch's column mean and *biased* column variance,
  and as a side effect `running_mean`/`running_var` each move `momentum` of the way toward
  the batch statistic, where the statistic tracked in `running_var` is the *unbiased* batch
  variance (torch's convention); that update is not part of the autograd graph. With
  `training=False`, `mean`/`var` are `running_mean`/`running_var` and neither is modified.
  Returns `out`, shape `(B, dim)`; in eval mode a batch of one example is valid input.
- `parameters()`: `[gamma, beta]`. The running stats are not in it.

The grader checks, in order: `parameters()` is two `(dim,)` tensors → gamma ones / beta zeros
→ `training is True` → five training-mode calls, each: output shape → (first call only)
column means ~0 and column stds ~1 → output equals torch's → gradient w.r.t. `x` equals
torch's → `running_mean` and `running_var` exist → `running_mean` equals torch's →
`running_var` equals torch's → buffers do not require grad → eval mode: output equals torch's
→ running stats untouched → a batch of one → gamma / beta actually scale and shift.

In [ ]:
class BatchNorm1d:
    """Batch normalization over a (B, dim) input.

    Fields:
      gamma, beta           (dim,) learnable scale and shift. Init ones / zeros.
      running_mean          (dim,) buffer, init zeros. Not a parameter, no grad.
      running_var           (dim,) buffer, init ones.  Not a parameter, no grad.
      training              bool, True at construction.
      out                   the last output (the plots read it).

    __call__(x), x: (B, dim)
      training=True:  normalize each column with THIS batch's mean and BIASED variance
                      (x.var(0, unbiased=False)); then update the running stats with
                      momentum: running = (1 - momentum) * running + momentum * batch_stat,
                      where the batch stat for running_var is the UNBIASED variance
                      (torch's convention). The running-stat update must not be part of
                      the autograd graph.
      training=False: normalize with running_mean / running_var. Do not touch them.
      always:         out = gamma * xhat + beta, with xhat = (x - mean) / sqrt(var + eps).

    parameters() -> [gamma, beta]
    """

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        raise NotImplementedError

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError

In [ ]:
grade(globals(), upto=4)

## Milestone 5 — pytorch-ify it, and it learns

Two small layer classes in the same style as your `BatchNorm1d`, so a model is just a list of
layers. The model assembly and the training loop are given below; the grader builds its own
model from your three classes, checks the loss at init, trains it for 400 steps, and evaluates
on dev with every layer in eval mode.

**Linear(fan_in, fan_out, bias=True, generator=None)**
- Holds: `weight` `(fan_in, fan_out)`, drawn with `torch.randn(..., generator=generator)`
  and scaled so a standard-normal `x` keeps its std (gain 1, so std `1/sqrt(fan_in)`);
  `bias` `(fan_out,)` zeros, or `None` when `bias=False`; `out`, the last output.
- `__call__(x)`, `x: (B, fan_in)`: `out = x @ weight + bias`, or `x @ weight` when there is
  no bias. Returns `out`, shape `(B, fan_out)`.
- `parameters()`: `[weight, bias]`, or `[weight]` when `bias=False`.

**Tanh()**
- Holds: `out`, the last output. No parameters.
- `__call__(x)`: `out = torch.tanh(x)`. Returns `out`, same shape as `x`.
- `parameters()`: `[]` (already written).

The grader checks, in order: `Linear(30, 100).parameters()` is `[weight, bias]` → shapes
`(30, 100)` and `(100,)` → bias is zero → weight std ≈ `1/sqrt(30)` → `lin(x) == x @ W + b`
→ `lin.out` is that output → `bias=False` gives one parameter → and computes `x @ W` →
`Tanh()(x) == torch.tanh(x)` → `t.out` is that output → `t.parameters() == []` → the assembled
model's loss at init is under 3.6 → after `backward()` every parameter has a grad → after
400 steps the loss averages under 2.75 → dev loss in eval mode is under 2.85.

In [ ]:
class Linear:
    """y = x @ weight + bias.

    weight: (fan_in, fan_out), drawn with torch.randn(..., generator=generator)
            and scaled so that a standard-normal x keeps its std (gain 1).
    bias:   (fan_out,) zeros, or None when bias=False.
    out:    the last output (the plots read it).
    parameters() -> [weight, bias] (or [weight] when there is no bias)
    """

    def __init__(self, fan_in, fan_out, bias=True, generator=None):
        raise NotImplementedError

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError


class Tanh:
    """Elementwise tanh. Keeps its last output in self.out. No parameters."""

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        return []

In [ ]:
grade(globals(), upto=5)

## Boilerplate: the deep model and the training loop

Given. `train()` runs a fixed number of steps and records, for every parameter, the
update-to-data ratio at each step (`ud`), so the diagnostic plots below have something to read.
`N_STEPS` is small so it takes seconds; raise it if you want the plots to look like the lecture's.

In [ ]:
N_HIDDEN_DEEP = 100


def make_model(g, use_bn=True, n_layers=5):
    C = torch.randn((vocab_size, N_EMBD), generator=g)
    layers = []
    fan_in = BLOCK * N_EMBD
    for _ in range(n_layers):
        layers.append(Linear(fan_in, N_HIDDEN_DEEP, bias=not use_bn, generator=g))
        if use_bn:
            layers.append(BatchNorm1d(N_HIDDEN_DEEP))
        layers.append(Tanh())
        fan_in = N_HIDDEN_DEEP
    layers.append(Linear(fan_in, vocab_size, bias=not use_bn, generator=g))
    if use_bn:
        layers.append(BatchNorm1d(vocab_size))
    with torch.no_grad():
        # make the last layer less confident
        layers[-1].parameters()[0] *= 0.1
        # boost the tanh layers' incoming weights by the tanh gain (only matters without BN)
        for layer in layers[:-1]:
            if isinstance(layer, Linear):
                layer.weight *= 5 / 3
    params = [C] + [p for layer in layers for p in layer.parameters()]
    for p in params:
        p.requires_grad = True
    print(sum(p.nelement() for p in params), 'parameters')
    return C, layers, params


def model_forward(C, layers, Xb):
    x = C[Xb].view(Xb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    return x


def train(C, layers, params, n_steps=1000, lr=0.1, batch_size=32, g=None, log_every=200):
    lossi, ud = [], []
    for i in range(n_steps):
        Xb, Yb = batch(Xtr, Ytr, batch_size, g)
        logits = model_forward(C, layers, Xb)
        loss = F.cross_entropy(logits, Yb)
        for layer in layers:
            layer.out.retain_grad()   # so the gradient plots can read layer.out.grad
        for p in params:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad
        if i % log_every == 0:
            print(f'{i:7d}/{n_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())
        with torch.no_grad():
            ud.append([((lr * p.grad).std() / p.std()).log10().item() for p in params])
    return lossi, ud


@torch.no_grad()
def split_loss(C, layers, split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    for layer in layers:
        layer.training = False
    loss = F.cross_entropy(model_forward(C, layers, X), Y)
    for layer in layers:
        layer.training = True
    print(split, f'{loss.item():.4f}')

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C, layers, params = make_model(g, use_bn=True)
lossi, ud = train(C, layers, params, n_steps=1000, g=g)

## Boilerplate: the four diagnostic plots

Given. Run them on the model above right after training (they read the gradients retained by the
last step, so run them *before* the eval cell below). Then try `make_model(g, use_bn=False)` and look again.

In [ ]:
def plot_activations(layers, kind='forward'):
    plt.figure(figsize=(16, 3))
    legends = []
    for i, layer in enumerate(layers[:-1]):
        if isinstance(layer, Tanh):
            t = layer.out if kind == 'forward' else layer.out.grad
            if kind == 'forward':
                print(f'layer {i} ({layer.__class__.__name__}): mean {t.mean():+.2f}, std {t.std():.2f}, saturated: {100*(t.abs() > 0.97).float().mean():.2f}%')
            else:
                print(f'layer {i} ({layer.__class__.__name__}): mean {t.mean():+f}, std {t.std():e}')
            hy, hx = torch.histogram(t.detach(), density=True)
            plt.plot(hx[:-1].detach(), hy.detach())
            legends.append(f'layer {i} ({layer.__class__.__name__})')
    plt.legend(legends); plt.title(f'{kind} pass distribution'); plt.show()


def plot_param_grads(params):
    plt.figure(figsize=(16, 3))
    legends = []
    for i, p in enumerate(params):
        t = p.grad
        if p.ndim == 2:
            print(f'weight {tuple(p.shape)} | mean {t.mean():+f} | std {t.std():e} | grad:data ratio {t.std() / p.std():e}')
            hy, hx = torch.histogram(t, density=True)
            plt.plot(hx[:-1].detach(), hy.detach())
            legends.append(f'{i} {tuple(p.shape)}')
    plt.legend(legends); plt.title('weights gradient distribution'); plt.show()


def plot_ud(params, ud):
    plt.figure(figsize=(16, 3))
    legends = []
    for i, p in enumerate(params):
        if p.ndim == 2:
            plt.plot([ud[j][i] for j in range(len(ud))])
            legends.append(f'param {i}')
    plt.plot([0, len(ud)], [-3, -3], 'k')   # the rule of thumb: ratios should be about 1e-3
    plt.legend(legends); plt.title('log10(update / data) per step'); plt.show()


plot_activations(layers, 'forward')
plot_activations(layers, 'backward')
plot_param_grads(params)
plot_ud(params, ud)

In [ ]:
split_loss(C, layers, 'train'); split_loss(C, layers, 'dev')

## Milestone 6 (stretch) — two diagnostics

`train()` above already computes the update-to-data ratio inline. Write it as a function, then a
second function that measures the lecture's throwaway claim: a bias feeding into a BatchNorm does
nothing. Nothing later in the notebook needs either function; the final grade cell skips this
milestone with `skip=(6,)`, so remove that argument once you have written them.

**update_to_data_ratio(p, lr)**
- `p`: a parameter tensor whose `.grad` is populated. `lr`: the learning rate.
- Computes `log10(std(lr * p.grad) / std(p))`. Returns a float; one number per parameter
  tensor, whatever its shape.

**bias_grad_through_bn(lin, bn, x)**
- `lin`: a `Linear` built with `bias=True`; `bn`: a `BatchNorm1d` in training mode;
  `x: (B, fan_in)`.
- Computes `bn(lin(x))`, reduces it to a scalar with a random weighting (so the check is not
  trivially zero), and backpropagates. Returns the gradient tensor that lands on `lin`'s bias,
  shape `(fan_out,)`, from a real backward pass. `lin`'s parameters must require grad by the
  time `backward()` runs. Layers not passed in are not touched.

The grader checks, in order: `update_to_data_ratio` on three tensors at scales 1, 0.01, 100
→ the weight ratios of the model trained in milestone 5 all sit in (-4.5, -1.0) →
`bias_grad_through_bn` returns a 64-entry tensor → its entries are all under 1e-5 in magnitude
→ a separate `Linear` that was never passed in still has `bias.grad is None`.

In [ ]:
def update_to_data_ratio(p, lr):
    """p: a parameter tensor whose .grad is populated. Return a float: log10 of
    (std of the update the optimizer is about to apply) / (std of p)."""
    raise NotImplementedError


def bias_grad_through_bn(lin, bn, x):
    """lin: a Linear with a bias; bn: a BatchNorm1d in training mode; x: (B, fan_in).
    Run x through lin then bn, reduce the output to a scalar with a random weighting
    (so the check isn't trivially zero), backprop, and return the gradient tensor
    that lands on lin's bias. Make sure the layer's parameters require grad."""
    raise NotImplementedError

In [ ]:
grade(globals(), skip=(6,))

## Your own training loop, and eval

From memory, without scrolling up: build a model from your layers, train it for a few hundred steps,
put it in eval mode, and report the dev loss. Then do it once more with `use_bn=False` and
without the `5/3` gain, and write down in the unit note what happened to the loss and to the plots.

In [ ]:
g = torch.Generator().manual_seed(1)

...

## Scratch

Space to poke at things. `forward_manual(params, Xb)` and `saturation_fraction(h)` are useful together.